In [0]:
-- =====================================================================
-- CPP + Visit-Level Concession Attach Rate by Population
-- CPP = Total Concession Revenue / Total Ticket Quantity
-- Attach Rate = Visits with concessions / All visits (visit_base grain)
-- Populations: CC Members | Non-CC Scene Members | Total Business (CPP only)
-- =====================================================================

WITH monthly_tickets AS (
    SELECT
        DATE_TRUNC('month', TO_DATE(CAST(t.VisitDateId AS STRING), 'yyyyMMdd')) AS visit_month,
        SUM(CASE WHEN t.CineClubFLAG = 'Yes' THEN t.Quantity ELSE 0 END)                                            AS cc_tickets,
        SUM(CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes' THEN t.Quantity ELSE 0 END)     AS scene_tickets,
        SUM(t.Quantity)                                                                                               AS total_tickets
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.TicketFLAG = 'Yes'
      AND t.VisitDateId >= 20240101
      AND t.VisitDateId <  CAST(DATE_FORMAT(DATE_TRUNC('month', CURRENT_DATE()), 'yyyyMMdd') AS INT)
    GROUP BY 1
),

monthly_concessions AS (
    SELECT
        DATE_TRUNC('month', TO_DATE(CAST(t.VisitDateId AS STRING), 'yyyyMMdd')) AS visit_month,
        SUM(CASE WHEN t.CineClubFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END)                                          AS cc_concession_rev,
        SUM(CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes' THEN t.Net_Revenue ELSE 0 END)   AS scene_concession_rev,
        SUM(t.Net_Revenue)                                                                                             AS total_concession_rev
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.ConcessionFLAG = 'Yes'
      AND t.VisitDateId >= 20240101
      AND t.VisitDateId <  CAST(DATE_FORMAT(DATE_TRUNC('month', CURRENT_DATE()), 'yyyyMMdd') AS INT)
    GROUP BY 1
),

-- Visit-level concession attach (verified visit_base pattern)
visit_base AS (
    SELECT
        CDE_ID, VisitDateId, L.LocationId,
        DATE_TRUNC('month', TO_DATE(CAST(VisitDateId AS STRING), 'yyyyMMdd')) AS visit_month,
        MAX(CineClubFLAG)                                                                         AS is_cineclub,
        MAX(CASE WHEN CineClubFLAG = 'Yes' AND ConcessionFLAG = 'Yes' THEN 'Yes' ELSE 'No' END)  AS has_cc_concession,
        MAX(CASE WHEN SCENEFLAG = 'Yes' AND ConcessionFLAG = 'Yes' THEN 'Yes' ELSE 'No' END)     AS has_scene_concession,
        MAX(CASE WHEN ConcessionFLAG = 'Yes' AND CineClub_Discount_Type_ID IN (2, 3) THEN 1 ELSE 0 END) AS has_disc_2_3
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction T
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location L ON T.LocationId = L.LocationID
    WHERE L.LC_Location_Type_Description = 'Cineplex Theatre'
      AND VisitDateId >= 20240101
      AND VisitDateId <  CAST(DATE_FORMAT(DATE_TRUNC('month', CURRENT_DATE()), 'yyyyMMdd') AS INT)
      AND (CineClubFLAG = 'Yes' OR SCENEFLAG = 'Yes')
    GROUP BY CDE_ID, VisitDateId, L.LocationId,
             DATE_TRUNC('month', TO_DATE(CAST(VisitDateId AS STRING), 'yyyyMMdd'))
),

visit_incidence AS (
    SELECT
        visit_month,
        -- CineClub
        COUNT(CASE WHEN is_cineclub = 'Yes' THEN 1 END)                                                              AS cc_visits,
        SUM(CASE WHEN is_cineclub = 'Yes' AND has_cc_concession = 'Yes' THEN 1 ELSE 0 END)                          AS cc_conc_visits,
        SUM(CASE WHEN is_cineclub = 'Yes' AND has_cc_concession = 'Yes' AND has_disc_2_3 = 1 THEN 1 ELSE 0 END)     AS cc_conc_disc_2_3,
        COUNT(DISTINCT CASE WHEN is_cineclub = 'Yes' THEN CDE_ID END)                                                AS cc_members,
        -- Scene (non-CC)
        COUNT(CASE WHEN is_cineclub != 'Yes' THEN 1 END)                                                             AS scene_visits,
        SUM(CASE WHEN is_cineclub != 'Yes' AND has_scene_concession = 'Yes' THEN 1 ELSE 0 END)                      AS scene_conc_visits,
        COUNT(DISTINCT CASE WHEN is_cineclub != 'Yes' THEN CDE_ID END)                                               AS scene_members
    FROM visit_base
    GROUP BY visit_month
)

SELECT
    tk.visit_month,
    YEAR(tk.visit_month)  AS yr,
    MONTH(tk.visit_month) AS mo,

    -- CPP by population
    ROUND(cn.cc_concession_rev    / NULLIF(tk.cc_tickets, 0), 2)                                  AS cc_cpp,
    ROUND(cn.scene_concession_rev / NULLIF(tk.scene_tickets, 0), 2)                               AS scene_cpp,
    ROUND(cn.total_concession_rev / NULLIF(tk.total_tickets, 0), 2)                               AS total_biz_cpp,
    ROUND(
        (cn.cc_concession_rev / NULLIF(tk.cc_tickets, 0))
      - (cn.scene_concession_rev / NULLIF(tk.scene_tickets, 0)), 2
    )                                                                                              AS cc_scene_cpp_gap,

    -- Visit-level concession attach rate (%)
    ROUND(vi.cc_conc_visits    * 100.0 / NULLIF(vi.cc_visits, 0), 1)                              AS cc_attach_rate_pct,
    ROUND(vi.scene_conc_visits * 100.0 / NULLIF(vi.scene_visits, 0), 1)                           AS scene_attach_rate_pct,

    -- CC discount 2/3 usage
    vi.cc_conc_disc_2_3,
    ROUND(vi.cc_conc_disc_2_3 * 100.0 / NULLIF(vi.cc_conc_visits, 0), 1)                          AS cc_disc_2_3_pct,

    -- Member & visit counts
    vi.cc_members,
    vi.scene_members,
    vi.cc_visits,
    vi.scene_visits,
    vi.cc_conc_visits,
    vi.scene_conc_visits,

    -- Ticket & revenue volumes
    tk.cc_tickets,
    tk.scene_tickets,
    tk.total_tickets,
    ROUND(cn.cc_concession_rev, 2)    AS cc_concession_rev,
    ROUND(cn.scene_concession_rev, 2) AS scene_concession_rev,
    ROUND(cn.total_concession_rev, 2) AS total_concession_rev

FROM monthly_tickets tk
JOIN monthly_concessions cn ON tk.visit_month = cn.visit_month
JOIN visit_incidence vi     ON tk.visit_month = vi.visit_month
ORDER BY tk.visit_month
